# 05 — Métricas, Segmentos y Parámetros para Tableau
## Smart Kitchen Intelligence (SKI) — Entrega Semana 11

**Curso:** Data Visualization — UPC  
**Fuente origen:** `data/interim/inventory_v1.csv` (25 819 eventos, flat table validada en Sem 5)  
**Fecha de ejecución:** 2026-06-13  

---
### Pregunta analítica
> **¿Qué hogares, categorías y momentos del periodo concentran el desperdicio alimentario, y qué métricas derivadas habilitan la lectura comparativa en Tableau sin recalcular en la herramienta?**

### Objetivos
1. Definir y calcular **métricas derivadas a nivel evento** con una sola fuente de verdad por métrica (sin duplicidad).
2. Construir **segmentos de hogar** (Eficiente / Promedio / Crítico) por terciles de tasa de desperdicio.
3. Materializar **parámetros analíticos** como tabla cruzable (umbrales de riesgo, terciles, buckets calóricos).
4. Exportar **fuentes finales** listas para conectar a Tableau **sin reprocesamiento manual**.
5. Documentar **chequeos de integridad** que garanticen cuadre cruzado entre fact, segmento y agregados.


## 1. Configuración e importaciones

In [ ]:
import os, json
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path('..').resolve()
SRC  = ROOT / 'data' / 'interim' / 'inventory_v1.csv'
OUT  = ROOT / 'tableau' / 'datos_finales'
OUT.mkdir(parents=True, exist_ok=True)
print('Fuente :', SRC)
print('Destino:', OUT)

## 2. Carga y validación de la fuente analítica

El dataset `inventory_v1.csv` es la *flat table* validada en la Semana 5: 25 819 eventos, 17 variables, sin duplicados de `event_id`. Esta es la **única** fuente que alimenta los cálculos de esta semana — no se vuelve a tocar el simulador ni el join con catálogo.

In [ ]:
df = pd.read_csv(SRC, parse_dates=['timestamp','expiry_date'])
assert df['event_id'].is_unique, 'PK event_id duplicado'
print(f'Filas: {len(df):,} | Cols: {df.shape[1]} | Rango: {df.timestamp.min().date()} → {df.timestamp.max().date()}')

## 3. Métricas derivadas a nivel evento

Cada métrica se materializa **una sola vez** como flag binario o bucket. Tableau debe sumar/contar — no recalcular reglas de negocio.

| Métrica | Regla | Tipo |
|---|---|---|
| `is_out` | `action_type == 'OUT'` | Binaria |
| `is_in` | `action_type == 'IN'` | Binaria |
| `is_waste` | `classification ∈ {Waste, Forced_Waste}` | Binaria |
| `is_forced_waste` | `classification == 'Forced_Waste'` | Binaria |
| `is_consumo` | `classification == 'Consumption'` | Binaria |
| `is_vencido` | `dias_para_vencer < 0` | Binaria |
| `flag_riesgo_vencer` | `0 ≤ dias_para_vencer ≤ 2` | Binaria (parámetro `umbral_riesgo_dias`) |
| `bucket_calorico` | corte sobre `calories_100g` en {50, 150, 300} | Ordinal |
| `turno` | corte sobre hora en {5,11,17,23} | Ordinal |

In [ ]:
df['is_out']           = (df['action_type'] == 'OUT').astype(int)
df['is_in']            = (df['action_type'] == 'IN').astype(int)
df['is_waste']         = df['classification'].isin(['Waste','Forced_Waste']).astype(int)
df['is_forced_waste']  = (df['classification'] == 'Forced_Waste').astype(int)
df['is_consumo']       = (df['classification'] == 'Consumption').astype(int)
df['is_vencido']       = (df['dias_para_vencer'] < 0).astype(int)
df['flag_riesgo_vencer'] = ((df['dias_para_vencer'] >= 0) & (df['dias_para_vencer'] <= 2)).astype(int)

df['fecha']      = df['timestamp'].dt.date
df['semana_iso'] = df['timestamp'].dt.strftime('%G-W%V')
df['mes']        = df['timestamp'].dt.to_period('M').astype(str)
df['dia_semana'] = df['timestamp'].dt.day_name()
df['hora']       = df['timestamp'].dt.hour
df['turno']      = pd.cut(df['hora'], bins=[-1,5,11,17,23],
                          labels=['Madrugada','Mañana','Tarde','Noche'])
df['bucket_calorico'] = pd.cut(df['calories_100g'],
                          bins=[-1,50,150,300,1000],
                          labels=['Bajo (<50)','Medio (50-150)','Alto (150-300)','Muy alto (>300)'])

df[['event_id','is_out','is_waste','is_vencido','flag_riesgo_vencer','turno','bucket_calorico']].head()

## 4. Fact table final para Tableau

PK = `event_id`. Granularidad = 1 fila por evento. Incluye TODAS las dimensiones y métricas binarias precomputadas. Tableau conecta este archivo como **fuente única** y deriva agregados con SUM()/COUNTD().

In [ ]:
fact_cols = ['event_id','household_id','product_id','product_name',
             'action_type','classification','quantity','timestamp','fecha',
             'semana_iso','mes','dia_semana','hora','turno',
             'expiry_date','dias_para_vencer','location','category_name',
             'nutriscore','calories_100g','proteins_100g','carbs_100g',
             'bucket_calorico',
             'is_out','is_in','is_waste','is_forced_waste','is_consumo',
             'is_vencido','flag_riesgo_vencer']
fact = df[fact_cols].copy()
fact.to_csv(OUT / 'fact_eventos_tableau.csv', index=False)
print(f'fact_eventos_tableau.csv → {len(fact):,} filas, {fact.shape[1]} columnas')

## 5. Segmentación de hogares (dim_hogar)

**Regla del segmento:**
- `tasa_desperdicio_eventos = eventos_waste / eventos_out` (sobre eventos OUT del hogar)
- Se calculan los terciles **Q33** y **Q66** sobre los 10 hogares.
- Segmento: `Eficiente` si tasa ≤ Q33, `Promedio` si Q33 < tasa ≤ Q66, `Crítico` si tasa > Q66.

**Por qué terciles y no umbrales absolutos:** el dataset es simulado; los terciles son robustos al nivel base de desperdicio y permiten comparación relativa sin asumir un benchmark externo.

**Métricas del hogar (todas únicas, no duplicadas con `fact`):**
`eventos_out`, `eventos_in`, `eventos_waste`, `eventos_forced`, `unidades_out`, `unidades_in`, `unidades_waste`, `tasa_desperdicio_eventos`, `tasa_desperdicio_unidades`, `tasa_forced_waste`, `segmento_hogar`, `umbral_q33`, `umbral_q66`.

In [ ]:
out = df[df['is_out']==1].copy()
agg_h = out.groupby('household_id').agg(
    eventos_out    = ('event_id','count'),
    eventos_waste  = ('is_waste','sum'),
    eventos_forced = ('is_forced_waste','sum'),
    unidades_out   = ('quantity','sum'),
    unidades_waste = ('quantity', lambda s: s[df.loc[s.index,'is_waste']==1].sum()),
).reset_index()

ins = df[df['is_in']==1].groupby('household_id').agg(
    eventos_in  = ('event_id','count'),
    unidades_in = ('quantity','sum'),
).reset_index()

dim_hogar = agg_h.merge(ins, on='household_id', how='left')
dim_hogar['tasa_desperdicio_eventos']  = (dim_hogar['eventos_waste']/dim_hogar['eventos_out']).round(4)
dim_hogar['tasa_desperdicio_unidades'] = (dim_hogar['unidades_waste']/dim_hogar['unidades_out']).round(4)
dim_hogar['tasa_forced_waste']         = (dim_hogar['eventos_forced']/dim_hogar['eventos_out']).round(4)

q33, q66 = dim_hogar['tasa_desperdicio_eventos'].quantile([0.33,0.66]).values
def seg(x):
    if x <= q33: return 'Eficiente'
    if x <= q66: return 'Promedio'
    return 'Crítico'
dim_hogar['segmento_hogar'] = dim_hogar['tasa_desperdicio_eventos'].apply(seg)
dim_hogar['umbral_q33'] = round(q33,4)
dim_hogar['umbral_q66'] = round(q66,4)

dim_hogar.to_csv(OUT / 'dim_hogar_segmentos.csv', index=False)
print(dim_hogar[['household_id','eventos_out','eventos_waste',
                 'tasa_desperdicio_eventos','segmento_hogar']].to_string(index=False))

## 6. Agregado transversal por categoría

Vista comparativa lista para Tableau. Permite leer **qué categoría concentra el desperdicio** sin que la herramienta tenga que cruzar la flat table cada vez.

In [ ]:
agg_cat = out.groupby('category_name', observed=True).agg(
    eventos_out    = ('event_id','count'),
    eventos_waste  = ('is_waste','sum'),
    unidades_out   = ('quantity','sum'),
    unidades_waste = ('quantity', lambda s: s[df.loc[s.index,'is_waste']==1].sum()),
    cal_promedio   = ('calories_100g','mean'),
).reset_index()
agg_cat['tasa_desperdicio'] = (agg_cat['eventos_waste']/agg_cat['eventos_out']).round(4)
agg_cat.to_csv(OUT / 'agg_categoria_metricas.csv', index=False)
print(agg_cat.to_string(index=False))

## 7. Agregado longitudinal semanal × categoría

Habilita la vista temporal del dashboard (línea de tendencia, comparativa semana a semana) sin que Tableau tenga que reagrupar 25 819 eventos.

In [ ]:
agg_t = out.groupby(['semana_iso','category_name'], observed=True).agg(
    eventos_out    = ('event_id','count'),
    eventos_waste  = ('is_waste','sum'),
    unidades_waste = ('quantity', lambda s: s[df.loc[s.index,'is_waste']==1].sum()),
).reset_index()
agg_t['tasa_desperdicio'] = (agg_t['eventos_waste']/agg_t['eventos_out']).round(4)
agg_t.to_csv(OUT / 'agg_temporal_semanal.csv', index=False)
print(f'{len(agg_t)} filas (semana × categoría)')
agg_t.head(8)

## 8. Tabla maestra de parámetros

Materializa todos los umbrales como datos cruzables — los **parámetros de Tableau** se enlazan a esta tabla (no se hardcodean en cálculos).

In [ ]:
parametros = pd.DataFrame([
    ['umbral_riesgo_dias',         2,            'int',   'Días ≤ valor → flag_riesgo_vencer'],
    ['umbral_segmento_q33',  round(q33,4),       'float', 'Tasa_desperdicio ≤ valor → Eficiente'],
    ['umbral_segmento_q66',  round(q66,4),       'float', 'Tasa_desperdicio ≤ valor → Promedio'],
    ['cutoff_calorico_bajo',       50,           'int',   'cal/100g ≤ valor → Bajo'],
    ['cutoff_calorico_medio',     150,           'int',   'cal/100g ≤ valor → Medio'],
    ['cutoff_calorico_alto',      300,           'int',   'cal/100g ≤ valor → Alto'],
    ['ventana_temporal_dias',      90,           'int',   'Cobertura del dataset (Feb-May 2026)'],
], columns=['parametro','valor','tipo','descripcion'])
parametros.to_csv(OUT / 'parametros_tableau.csv', index=False)
parametros

## 9. QA — Cuadre cruzado de integridad

Antes de publicar las fuentes, validamos que:
- PKs únicas en `fact` y `dim_hogar`.
- El total de eventos OUT cuadra entre `fact`, `dim_hogar` y `agg_categoria`.
- El total de eventos waste cuadra entre las 3 tablas.

Si algún `assert` falla, **NO** se exportan las fuentes y se revisa la regla.

In [ ]:
checks = {
    'fact_filas': len(fact),
    'fact_pk_unica': bool(fact['event_id'].is_unique),
    'dim_hogar_pk_unica': bool(dim_hogar['household_id'].is_unique),
    'eventos_out_fact':  int(fact['is_out'].sum()),
    'eventos_out_dim_hogar': int(dim_hogar['eventos_out'].sum()),
    'eventos_out_agg_cat':   int(agg_cat['eventos_out'].sum()),
    'eventos_waste_fact':    int(fact['is_waste'].sum()),
    'eventos_waste_dim_hogar': int(dim_hogar['eventos_waste'].sum()),
    'eventos_waste_agg_cat':   int(agg_cat['eventos_waste'].sum()),
}
checks['cuadre_eventos_OUT'] = (checks['eventos_out_fact'] == checks['eventos_out_dim_hogar'] == checks['eventos_out_agg_cat'])
checks['cuadre_eventos_waste'] = (checks['eventos_waste_fact'] == checks['eventos_waste_dim_hogar'] == checks['eventos_waste_agg_cat'])
assert checks['cuadre_eventos_OUT'],   'Falla cuadre OUT'
assert checks['cuadre_eventos_waste'], 'Falla cuadre waste'
with open(OUT/'qa_integridad.json','w') as f:
    json.dump(checks, f, indent=2, default=str)
print(json.dumps(checks, indent=2, default=str))

## 10. Resumen y entregables

| Archivo | Contenido | Granularidad | Uso en Tableau |
|---|---|---|---|
| `fact_eventos_tableau.csv` | 25 819 eventos con métricas binarias y buckets | 1 fila/evento | Fuente principal — todas las vistas detalle |
| `dim_hogar_segmentos.csv` | 10 hogares con KPIs y segmento | 1 fila/hogar | Filtro por segmento, ranking de hogares |
| `agg_categoria_metricas.csv` | KPIs por categoría | 1 fila/categoría | Bloque transversal del dashboard |
| `agg_temporal_semanal.csv` | KPIs por semana × categoría | 1 fila/semana×cat | Bloque longitudinal |
| `parametros_tableau.csv` | Umbrales materializados | 1 fila/parámetro | Tabla maestra de parámetros |
| `qa_integridad.json` | Chequeos de cuadre | — | Evidencia de validación |

**Conexión recomendada en Tableau:**
- `fact_eventos_tableau.csv` como fuente principal.
- `dim_hogar_segmentos.csv` relacionado por `household_id` (1:N).
- `agg_categoria` y `agg_temporal` como **fuentes independientes** para los bloques agregados (evita doble conteo si se unieran al fact).
- `parametros_tableau.csv` como hoja de referencia para sincronizar parámetros del workbook con la lógica del pipeline.

Ver `docs/reglas_metricas_segmentos.md` para la especificación funcional completa.